# Memory + RAG Integration

Combines VectorStore retrieval with modern LCEL conversational memory. Uses create_history_aware_retriever to rephrase follow-up queries using chat history before vector search, powered by gemini-3.6-flash.

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_community.chat_message_histories import ChatMessageHistory

load_dotenv(find_dotenv())

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

docs = [
    Document(page_content="The Apollo 11 mission landed humans on the Moon on July 20, 1969. Neil Armstrong was commander."),
    Document(page_content="Buzz Aldrin was the Lunar Module Pilot for Apollo 11. Michael Collins flew the Command Module.")
]

vectorstore = Chroma.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

# History-aware retriever prompt
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", "Formulate a standalone question from chat history if needed. Do NOT answer it."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])
history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_q_prompt)

# QA prompt
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using only context:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])
qa_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, qa_chain)

# Modern LCEL Memory Execution Helper
history = ChatMessageHistory()

def ask_conversational_rag(user_query: str):
    res = rag_chain.invoke({"input": user_query, "chat_history": history.messages})
    history.add_user_message(user_query)
    history.add_ai_message(res["answer"])
    return res["answer"]

print("Memory + RAG pipeline initialized successfully with gemini-3.6-flash!")


Memory + RAG pipeline initialized successfully with gemini-3.6-flash!
